In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [14]:
corpus = [
    "dota hero draft",
    "ancient creep camp",
    "radiant bottom lane",
    "dire middle tower",
    "carry farming jungle",
    "support ward placement",
    "roshan pit fight",
    "black king bar",
    "blink dagger initiation",
    "shadow blade escape",
    "aghanim scepter upgrade",
    "butterfly damage item",
    "manta style illusion",
    "divine rapier damage",
    "creep wave control",
    "mid lane rotation",
    "offlane pressure strategy",
    "smoke gank timing",
    "team fight coordination",
    "base defense glyph",
]

In [15]:
#1. tokenization
corpus = [sent.split(" ") for sent in corpus]
corpus

[['dota', 'hero', 'draft'],
 ['ancient', 'creep', 'camp'],
 ['radiant', 'bottom', 'lane'],
 ['dire', 'middle', 'tower'],
 ['carry', 'farming', 'jungle'],
 ['support', 'ward', 'placement'],
 ['roshan', 'pit', 'fight'],
 ['black', 'king', 'bar'],
 ['blink', 'dagger', 'initiation'],
 ['shadow', 'blade', 'escape'],
 ['aghanim', 'scepter', 'upgrade'],
 ['butterfly', 'damage', 'item'],
 ['manta', 'style', 'illusion'],
 ['divine', 'rapier', 'damage'],
 ['creep', 'wave', 'control'],
 ['mid', 'lane', 'rotation'],
 ['offlane', 'pressure', 'strategy'],
 ['smoke', 'gank', 'timing'],
 ['team', 'fight', 'coordination'],
 ['base', 'defense', 'glyph']]

In [16]:
#2. numeralization
#find unique words
flatten = lambda l: [item for sublist in l for item in sublist]
#assign unique integer
vocabs = list(set(flatten(corpus)))
vocabs

['hero',
 'damage',
 'black',
 'camp',
 'draft',
 'team',
 'escape',
 'offlane',
 'radiant',
 'ancient',
 'illusion',
 'pressure',
 'initiation',
 'tower',
 'creep',
 'strategy',
 'gank',
 'aghanim',
 'shadow',
 'wave',
 'base',
 'smoke',
 'bottom',
 'timing',
 'dagger',
 'scepter',
 'manta',
 'defense',
 'ward',
 'pit',
 'dire',
 'butterfly',
 'upgrade',
 'fight',
 'dota',
 'farming',
 'blade',
 'jungle',
 'rapier',
 'mid',
 'roshan',
 'coordination',
 'divine',
 'support',
 'style',
 'item',
 'carry',
 'blink',
 'king',
 'middle',
 'bar',
 'glyph',
 'control',
 'lane',
 'rotation',
 'placement']

In [17]:
#create handy mapping between integer and word
word2index = {v:idx for idx, v in enumerate(vocabs)}
word2index["placement"]

55

In [18]:
vocabs.append("<UNK>")
word2index["<UNK>"] = 56
word2index["<UNK>"]

56

In [19]:
index2word = {v: k for k, v in word2index.items()}
index2word

{0: 'hero',
 1: 'damage',
 2: 'black',
 3: 'camp',
 4: 'draft',
 5: 'team',
 6: 'escape',
 7: 'offlane',
 8: 'radiant',
 9: 'ancient',
 10: 'illusion',
 11: 'pressure',
 12: 'initiation',
 13: 'tower',
 14: 'creep',
 15: 'strategy',
 16: 'gank',
 17: 'aghanim',
 18: 'shadow',
 19: 'wave',
 20: 'base',
 21: 'smoke',
 22: 'bottom',
 23: 'timing',
 24: 'dagger',
 25: 'scepter',
 26: 'manta',
 27: 'defense',
 28: 'ward',
 29: 'pit',
 30: 'dire',
 31: 'butterfly',
 32: 'upgrade',
 33: 'fight',
 34: 'dota',
 35: 'farming',
 36: 'blade',
 37: 'jungle',
 38: 'rapier',
 39: 'mid',
 40: 'roshan',
 41: 'coordination',
 42: 'divine',
 43: 'support',
 44: 'style',
 45: 'item',
 46: 'carry',
 47: 'blink',
 48: 'king',
 49: 'middle',
 50: 'bar',
 51: 'glyph',
 52: 'control',
 53: 'lane',
 54: 'rotation',
 55: 'placement',
 56: '<UNK>'}

In [21]:
#create pairs of center word, and outside word

#loop each corpus
    #loop each document
        # look from the 2nd word until second last word (window = 1)
            # center word
            # outside words = 2 words
            # for each of these two outside words, we gonna append to a list
                # center, outside1; center, outside2
skipgrams = []
for doc in corpus:
    for i in range(1, len(doc) - 1):
        center = word2index[doc[i]]
        outside = (word2index[doc[i - 1]], word2index[doc[i + 1]])
        for each_out in outside:
            skipgrams.append([center, each_out])
skipgrams

[[0, 34],
 [0, 4],
 [14, 9],
 [14, 3],
 [22, 8],
 [22, 53],
 [49, 30],
 [49, 13],
 [35, 46],
 [35, 37],
 [28, 43],
 [28, 55],
 [29, 40],
 [29, 33],
 [48, 2],
 [48, 50],
 [24, 47],
 [24, 12],
 [36, 18],
 [36, 6],
 [25, 17],
 [25, 32],
 [1, 31],
 [1, 45],
 [44, 26],
 [44, 10],
 [38, 42],
 [38, 1],
 [19, 14],
 [19, 52],
 [53, 39],
 [53, 54],
 [11, 7],
 [11, 15],
 [16, 21],
 [16, 23],
 [33, 5],
 [33, 41],
 [27, 20],
 [27, 51]]

In [27]:
def random_batch(batch_size, corpus):
    skipgrams = []
    for doc in corpus:
        for i in range(1, len(doc) - 1):
            center = word2index[doc[i]]
            outside = (word2index[doc[i - 1]], word2index[doc[i + 1]])
            for each_out in outside:
                skipgrams.append([center, each_out])

    random_indexes = np.random.choice(range(len(skipgrams)), batch_size, replace=False)
    inputs, labels = [], []
    for index in random_indexes:
        inputs.append([skipgrams[index][0]])
        labels.append([skipgrams[index][1]])
    return np.array(inputs), np.array(labels)

x, y = random_batch(3, corpus)
x.shape
y.shape
x, y

(array([[33],
        [35],
        [35]]),
 array([[ 5],
        [37],
        [46]]))

In [ ]:
# model

class Skipgram(nn.Module):
    def __init__(self):
        pass

    def forward(self, center, outside, all_vocabs):
        pass